# Mortality: whole-cohort out-of-fold risk scores

Using the hyperparameters selected in notebook 1, fit each model in 5-fold cross-validation over the full cohort. Every patient receives a prediction from a model that did not train on that patient, yielding the six requested risk-score vectors.

In [ ]:
from pathlib import Path
import json
import os
import sys

import pandas as pd

def find_repo_root(start=Path.cwd()):
    for candidate in [start, *start.parents]:
        if (candidate / 'python_scripts' / 'model_training' / 'slurm_array_utils.py').exists():
            return candidate
    raise FileNotFoundError('Could not locate the clinical_text_embedding_project repository root.')

REPO_ROOT = find_repo_root()
NOTEBOOK_DIR = REPO_ROOT / 'jupyter_notebooks' / 'mortality_model_comparison'
sys.path.insert(0, str(NOTEBOOK_DIR))
sys.path.insert(0, str(REPO_ROOT / 'python_scripts' / 'model_training'))

from slurm_array_utils import SURV_PATH, build_full_prediction_df, filter_event_rows
from survival_benchmark import generate_oof_risk_scores, prepare_cohort

In [ ]:
RANDOM_STATE = 1234
N_JOBS = int(os.getenv('SLURM_CPUS_PER_TASK', '1'))
OUTPUT_DIR = Path(SURV_PATH) / 'results' / 'death_met_results' / 'mortality_model_comparison'
BEST_PARAMS_PATH = OUTPUT_DIR / 'best_hyperparameters.json'
if not BEST_PARAMS_PATH.exists():
    raise FileNotFoundError(f'Run 01_tune_and_test.ipynb first: {BEST_PARAMS_PATH}')
with BEST_PARAMS_PATH.open() as handle:
    best_hyperparameters = json.load(handle)
best_hyperparameters

## Rebuild and verify the cohort

The patient identifiers must exactly match the cohort recorded by notebook 1 before cross-fitting begins.

In [ ]:
full_df, cancer_type_cols, text_cols, events = build_full_prediction_df('death_met')
mortality_df = filter_event_rows(full_df, 'death')
baseline_cols = ['GENDER', 'AGE_AT_TREATMENTSTART'] + cancer_type_cols
cohort, feature_sets = prepare_cohort(
    mortality_df,
    baseline_cols=baseline_cols,
    text_cols=text_cols,
)

recorded_ids = set(pd.read_csv(OUTPUT_DIR / 'train_test_split.csv')['DFCI_MRN'])
current_ids = set(cohort['DFCI_MRN'])
if current_ids != recorded_ids:
    raise RuntimeError(
        f'Cohort changed after tuning: {len(recorded_ids - current_ids)} removed and '
        f'{len(current_ids - recorded_ids)} added. Re-run notebook 1.'
    )
len(cohort), int(cohort['death'].sum())

## Generate six held-out risk-score sets

The wide output has one row per patient and six risk columns named `<model>__<feature_set>`. Scores are standardized fold-by-fold using only training-fold predictions, which avoids arbitrary shifts between separately fitted models without using held-out outcomes.

In [ ]:
risk_scores, oof_timing = generate_oof_risk_scores(
    cohort,
    feature_sets,
    best_hyperparameters,
    OUTPUT_DIR,
    n_splits=5,
    random_state=RANDOM_STATE,
    n_jobs=N_JOBS,
)
risk_cols = [c for c in risk_scores if '__' in c]
assert len(risk_cols) == 6
assert risk_scores[risk_cols].notna().all().all()
risk_scores.head()

In [ ]:
display(risk_scores[risk_cols].describe().T)
display(oof_timing.query("fold == 'all'").sort_values(['model', 'feature_set']))
print(f'Wide risk-score file: {OUTPUT_DIR / "mortality_oof_risk_scores.csv"}')